# SaaS Churn Prediction Model

**Goal:** Train and evaluate a binary classifier to predict whether a customer will churn in the next month.

**Approach:**
- **Observation date:** Jan 31 2026 -- features computed from all data up to this date
- **Label:** `churned = 1` if a user had orders before Feb 2026 but placed zero orders in Feb
- **Label:** `churned = 0` if they ordered in both periods (renewed)
- **Tracking:** MLflow experiment logged to Databricks workspace

| | Value |
|--|--|
| Feature table | `saas_analysis_sandbox.churn_features` |
| Training users | 7,169 (52.8% retained, 47.2% churned) |
| Feature count | 22 numeric + 3 categorical |
| Models compared | Logistic Regression, Random Forest, LightGBM, XGBoost |

## 1. Setup

In [7]:
from databricks.connect import DatabricksSession
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

spark = DatabricksSession.builder.serverless(True).profile("DEFAULT").getOrCreate()

CATALOG = "sv_ai_builder_workspace_catalog"
GOLD    = "saas_gold_dev"
SANDBOX = "saas_analysis_sandbox"

import mlflow
mlflow.set_tracking_uri("databricks")
EXPERIMENT = "/Users/saroj.venkatesh@databricks.com/saas_churn_prediction"
mlflow.set_experiment(EXPERIMENT)

print(f"Spark         : {spark.version}")
print(f"MLflow        : {mlflow.__version__}")
print(f"Experiment    : {EXPERIMENT}")

Spark         : 4.1.0
MLflow        : 3.10.0
Experiment    : /Users/saroj.venkatesh@databricks.com/saas_churn_prediction


## 2. Build Feature Table

One row per user. Features computed from data **before** Feb 1 2026. Label = did they order in Feb?

**Feature groups:**
- **Recency/Frequency/Monetary** -- order history signals (strongest predictors)
- **Momentum** -- order trend (last 30d vs prior 30d)
- **Product experience** -- error rate, latency from event logs
- **Account attributes** -- tier, product, geography, enterprise, MFA, tenure

In [8]:
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.{SANDBOX}.churn_features AS
WITH eligible_users AS (
    SELECT DISTINCT user_id FROM {CATALOG}.{GOLD}.fact_order
    WHERE order_date < '2026-02-01' AND status != 'cancelled'
),
churn_label AS (
    SELECT e.user_id,
           CASE WHEN feb.user_id IS NULL THEN 1 ELSE 0 END AS churned
    FROM eligible_users e
    LEFT JOIN (
        SELECT DISTINCT user_id FROM {CATALOG}.{GOLD}.fact_order
        WHERE order_date >= '2026-02-01' AND status != 'cancelled'
    ) feb ON e.user_id = feb.user_id
),
order_features AS (
    SELECT user_id,
           DATEDIFF(DATE('2026-01-31'), MAX(order_date))                                AS days_since_last_order,
           COUNT(order_id)                                                               AS lifetime_orders,
           CAST(ROUND(AVG(total_value), 2) AS DOUBLE)                                   AS avg_order_value,
           CAST(ROUND(SUM(total_value), 2) AS DOUBLE)                                   AS total_ltv,
           SUM(CASE WHEN status = 'refunded' THEN 1 ELSE 0 END)                         AS refund_count,
           COUNT(CASE WHEN order_date >= DATE_SUB(DATE('2026-01-31'), 30)
                       THEN order_id END)                                                AS orders_last_30d,
           COUNT(CASE WHEN order_date BETWEEN DATE_SUB(DATE('2026-01-31'), 60)
                                         AND DATE_SUB(DATE('2026-01-31'), 31)
                       THEN order_id END)                                                AS orders_prev_30d
    FROM {CATALOG}.{GOLD}.fact_order
    WHERE status != 'cancelled' AND order_date < '2026-02-01'
    GROUP BY user_id
),
event_features AS (
    SELECT user_id,
           COUNT(event_id)                                                                           AS total_events,
           COUNT(DISTINCT session_id)                                                                AS total_sessions,
           CAST(ROUND(100.0 * SUM(CASE WHEN is_error_event  THEN 1 ELSE 0 END) / COUNT(*), 2) AS DOUBLE) AS error_rate_pct,
           CAST(ROUND(100.0 * SUM(CASE WHEN is_high_latency THEN 1 ELSE 0 END) / COUNT(*), 2) AS DOUBLE) AS high_latency_pct,
           CAST(ROUND(AVG(latency_ms), 0) AS DOUBLE)                                                     AS avg_latency_ms,
           CAST(ROUND(100.0 * SUM(CASE WHEN event_category = 'feature_use' THEN 1 ELSE 0 END) / COUNT(*), 2) AS DOUBLE) AS feature_use_rate,
           CAST(ROUND(100.0 * SUM(CASE WHEN event_category = 'api_call'    THEN 1 ELSE 0 END) / COUNT(*), 2) AS DOUBLE) AS api_use_rate,
           COUNT(CASE WHEN event_date >= DATE_SUB(DATE('2026-01-31'), 30) THEN event_id END)        AS events_last_30d
    FROM {CATALOG}.{GOLD}.fact_event
    WHERE event_date < '2026-02-01'
    GROUP BY user_id
),
user_dim AS (
    SELECT user_id, subscription_tier, product_type, country,
           CAST(is_enterprise AS INT) AS is_enterprise,
           CAST(mfa_enabled   AS INT) AS mfa_enabled,
           COALESCE(account_age_days, 0) AS account_age_days
    FROM {CATALOG}.{GOLD}.dim_user WHERE __END_AT IS NULL
)
SELECT c.user_id, c.churned,
       COALESCE(o.days_since_last_order, 999)                           AS days_since_last_order,
       COALESCE(o.lifetime_orders,  0)                                  AS lifetime_orders,
       COALESCE(o.avg_order_value,  0.0)                                AS avg_order_value,
       COALESCE(o.total_ltv,        0.0)                                AS total_ltv,
       COALESCE(o.refund_count,     0)                                  AS refund_count,
       COALESCE(o.orders_last_30d,  0)                                  AS orders_last_30d,
       COALESCE(o.orders_prev_30d,  0)                                  AS orders_prev_30d,
       COALESCE(o.orders_last_30d, 0) - COALESCE(o.orders_prev_30d, 0) AS order_momentum,
       COALESCE(e.total_events,     0)                                  AS total_events,
       COALESCE(e.total_sessions,   0)                                  AS total_sessions,
       COALESCE(e.error_rate_pct,   0.0)                                AS error_rate_pct,
       COALESCE(e.high_latency_pct, 0.0)                                AS high_latency_pct,
       COALESCE(e.avg_latency_ms,   0.0)                                AS avg_latency_ms,
       COALESCE(e.feature_use_rate, 0.0)                                AS feature_use_rate,
       COALESCE(e.api_use_rate,     0.0)                                AS api_use_rate,
       COALESCE(e.events_last_30d,  0)                                  AS events_last_30d,
       COALESCE(u.subscription_tier, 'Unknown')                         AS subscription_tier,
       COALESCE(u.product_type,      'Unknown')                         AS product_type,
       COALESCE(u.country,           'Unknown')                         AS country,
       COALESCE(u.is_enterprise,     0)                                  AS is_enterprise,
       COALESCE(u.mfa_enabled,       0)                                  AS mfa_enabled,
       COALESCE(u.account_age_days,  0)                                  AS account_age_days
FROM churn_label c
LEFT JOIN order_features o ON c.user_id = o.user_id
LEFT JOIN event_features e ON c.user_id = e.user_id
LEFT JOIN user_dim        u ON c.user_id = u.user_id
""")

spark.sql(f"""
    SELECT churned, COUNT(*) AS users,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM {CATALOG}.{SANDBOX}.churn_features GROUP BY churned ORDER BY churned
""").show()
print(f"Feature table saved: {CATALOG}.{SANDBOX}.churn_features")

+-------+-----+----+
|churned|users| pct|
+-------+-----+----+
|      0| 3783|52.8|
|      1| 3386|47.2|
+-------+-----+----+

Feature table saved: sv_ai_builder_workspace_catalog.saas_analysis_sandbox.churn_features


## 3. Prepare Training Data

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

df = spark.table(f"{CATALOG}.{SANDBOX}.churn_features").toPandas()

# Spark DECIMAL types come through as Python Decimal objects (dtype=object).
# Coerce all non-categorical, non-id columns to float64 to be safe.
CAT_COLS  = ['subscription_tier', 'product_type', 'country']
DROP_COLS = ['user_id', 'churned']
NUM_COLS  = [c for c in df.columns if c not in CAT_COLS + DROP_COLS]

# Cast any Decimal/object numerics to float64
df[NUM_COLS] = df[NUM_COLS].apply(lambda s: s.astype(float))

enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
df[CAT_COLS] = enc.fit_transform(df[CAT_COLS])

FEATURE_COLS = NUM_COLS + CAT_COLS
X = df[FEATURE_COLS]
y = df['churned']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print(f"Train : {len(X_train):,} rows  (churn rate: {y_train.mean():.1%})")
print(f"Test  : {len(X_test):,}  rows  (churn rate: {y_test.mean():.1%})")
print(f"Features ({len(FEATURE_COLS)}): {', '.join(FEATURE_COLS)}")

Train : 5,735 rows  (churn rate: 47.2%)
Test  : 1,434  rows  (churn rate: 47.2%)
Features (22): days_since_last_order, lifetime_orders, avg_order_value, total_ltv, refund_count, orders_last_30d, orders_prev_30d, order_momentum, total_events, total_sessions, error_rate_pct, high_latency_pct, avg_latency_ms, feature_use_rate, api_use_rate, events_last_30d, is_enterprise, mfa_enabled, account_age_days, subscription_tier, product_type, country


## 4. Train & Compare Models

All runs logged to the MLflow experiment. Metric: **ROC-AUC** (best for imbalanced churn classification).
PR-AUC is also tracked as it penalises false negatives (missed churn) more heavily.

In [10]:
from sklearn.linear_model  import LogisticRegression
from sklearn.ensemble       import RandomForestClassifier
from lightgbm               import LGBMClassifier
from xgboost                import XGBClassifier
from sklearn.metrics        import roc_auc_score, average_precision_score, classification_report
from mlflow.models          import infer_signature
import mlflow.sklearn

MODELS = {
    "logistic_regression": LogisticRegression(max_iter=500, random_state=42),
    "random_forest":       RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1),
    "lightgbm":            LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=63,
                                          random_state=42, verbose=-1),
    "xgboost":             XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                         random_state=42, verbosity=0),
}

results = {}

for name, model in MODELS.items():
    with mlflow.start_run(run_name=name):
        mlflow.set_tag("model_type", name)
        mlflow.set_tag("dataset",    f"{CATALOG}.{SANDBOX}.churn_features")

        model.fit(X_train, y_train)
        y_prob = model.predict_proba(X_test)[:, 1]

        auc    = roc_auc_score(y_test, y_prob)
        pr_auc = average_precision_score(y_test, y_prob)

        signature = infer_signature(X_train, y_prob)
        mlflow.log_params({"n_features": len(FEATURE_COLS), "test_rows": len(X_test)})
        mlflow.log_metrics({"roc_auc": round(auc, 4), "pr_auc": round(pr_auc, 4)})
        mlflow.sklearn.log_model(model, name=name, signature=signature)

        results[name] = {"roc_auc": auc, "pr_auc": pr_auc, "model": model}
        print(f"{name:<25}  ROC-AUC: {auc:.4f}   PR-AUC: {pr_auc:.4f}")

best_name  = max(results, key=lambda k: results[k]['roc_auc'])
best_model = results[best_name]['model']
print(f"\nBest model: {best_name}  (ROC-AUC: {results[best_name]['roc_auc']:.4f})")
print(f"\nClassification report ({best_name} @ 0.5 threshold):")
print(classification_report(y_test, best_model.predict(X_test), target_names=['Retained','Churned']))

{"ts": "2026-02-26 14:53:45,478", "level": "ERROR", "logger": "pyspark.sql.connect.client.logging", "msg": "GRPC Error received", "context": {}, "exception": {"class": "_InactiveRpcError", "msg": "<_InactiveRpcError of RPC that terminated with:\n\tstatus = StatusCode.INTERNAL\n\tdetails = \"[CONFIG_NOT_AVAILABLE] Configuration spark.mlflow.modelRegistryUri is not available. SQLSTATE: 42K0I\"\n\tdebug_error_string = \"UNKNOWN:Error received from peer  {grpc_status:13, grpc_message:\"[CONFIG_NOT_AVAILABLE] Configuration spark.mlflow.modelRegistryUri is not available. SQLSTATE: 42K0I\"}\"\n>", "stacktrace": ["Traceback (most recent call last):", "  File \"/Users/saroj.venkatesh/ai-dev-kit/.venv/lib/python3.11/site-packages/pyspark/sql/connect/client/core.py\", line 2042, in config", "    resp = self._stub.Config(req, metadata=self.metadata())", "           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^", "  File \"/Users/saroj.venkatesh/ai-dev-kit/.venv/lib/python3.11/site-packages/grpc

logistic_regression        ROC-AUC: 0.7241   PR-AUC: 0.6596


{"ts": "2026-02-26 14:54:14,782", "level": "ERROR", "logger": "pyspark.sql.connect.client.logging", "msg": "GRPC Error received", "context": {}, "exception": {"class": "_InactiveRpcError", "msg": "<_InactiveRpcError of RPC that terminated with:\n\tstatus = StatusCode.INTERNAL\n\tdetails = \"[CONFIG_NOT_AVAILABLE] Configuration spark.mlflow.modelRegistryUri is not available. SQLSTATE: 42K0I\"\n\tdebug_error_string = \"UNKNOWN:Error received from peer  {grpc_status:13, grpc_message:\"[CONFIG_NOT_AVAILABLE] Configuration spark.mlflow.modelRegistryUri is not available. SQLSTATE: 42K0I\"}\"\n>", "stacktrace": ["Traceback (most recent call last):", "  File \"/Users/saroj.venkatesh/ai-dev-kit/.venv/lib/python3.11/site-packages/pyspark/sql/connect/client/core.py\", line 2042, in config", "    resp = self._stub.Config(req, metadata=self.metadata())", "           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^", "  File \"/Users/saroj.venkatesh/ai-dev-kit/.venv/lib/python3.11/site-packages/grpc

🏃 View run logistic_regression at: https://fevm-sv-ai-builder-workspace.cloud.databricks.com/ml/experiments/125118335144552/runs/420c8cfce943490f892b7dbf35ee33b5
🧪 View experiment at: https://fevm-sv-ai-builder-workspace.cloud.databricks.com/ml/experiments/125118335144552


{"ts": "2026-02-26 14:54:19,022", "level": "ERROR", "logger": "pyspark.sql.connect.client.logging", "msg": "GRPC Error received", "context": {}, "exception": {"class": "_InactiveRpcError", "msg": "<_InactiveRpcError of RPC that terminated with:\n\tstatus = StatusCode.INTERNAL\n\tdetails = \"[CONFIG_NOT_AVAILABLE] Configuration spark.mlflow.modelRegistryUri is not available. SQLSTATE: 42K0I\"\n\tdebug_error_string = \"UNKNOWN:Error received from peer  {grpc_message:\"[CONFIG_NOT_AVAILABLE] Configuration spark.mlflow.modelRegistryUri is not available. SQLSTATE: 42K0I\", grpc_status:13}\"\n>", "stacktrace": ["Traceback (most recent call last):", "  File \"/Users/saroj.venkatesh/ai-dev-kit/.venv/lib/python3.11/site-packages/pyspark/sql/connect/client/core.py\", line 2042, in config", "    resp = self._stub.Config(req, metadata=self.metadata())", "           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^", "  File \"/Users/saroj.venkatesh/ai-dev-kit/.venv/lib/python3.11/site-packages/grpc

...

🏃 View run lightgbm at: https://fevm-sv-ai-builder-workspace.cloud.databricks.com/ml/experiments/125118335144552/runs/27d862b1a6c440e0bd5de021aa17fcc7
🧪 View experiment at: https://fevm-sv-ai-builder-workspace.cloud.databricks.com/ml/experiments/125118335144552


{"ts": "2026-02-26 14:55:32,659", "level": "ERROR", "logger": "pyspark.sql.connect.client.logging", "msg": "GRPC Error received", "context": {}, "exception": {"class": "_InactiveRpcError", "msg": "<_InactiveRpcError of RPC that terminated with:\n\tstatus = StatusCode.INTERNAL\n\tdetails = \"[CONFIG_NOT_AVAILABLE] Configuration spark.mlflow.modelRegistryUri is not available. SQLSTATE: 42K0I\"\n\tdebug_error_string = \"UNKNOWN:Error received from peer  {grpc_status:13, grpc_message:\"[CONFIG_NOT_AVAILABLE] Configuration spark.mlflow.modelRegistryUri is not available. SQLSTATE: 42K0I\"}\"\n>", "stacktrace": ["Traceback (most recent call last):", "  File \"/Users/saroj.venkatesh/ai-dev-kit/.venv/lib/python3.11/site-packages/pyspark/sql/connect/client/core.py\", line 2042, in config", "    resp = self._stub.Config(req, metadata=self.metadata())", "           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^", "  File \"/Users/saroj.venkatesh/ai-dev-kit/.venv/lib/python3.11/site-packages/grpc

xgboost                    ROC-AUC: 0.7422   PR-AUC: 0.7018


{"ts": "2026-02-26 14:56:03,999", "level": "ERROR", "logger": "pyspark.sql.connect.client.logging", "msg": "GRPC Error received", "context": {}, "exception": {"class": "_InactiveRpcError", "msg": "<_InactiveRpcError of RPC that terminated with:\n\tstatus = StatusCode.INTERNAL\n\tdetails = \"[CONFIG_NOT_AVAILABLE] Configuration spark.mlflow.modelRegistryUri is not available. SQLSTATE: 42K0I\"\n\tdebug_error_string = \"UNKNOWN:Error received from peer  {grpc_message:\"[CONFIG_NOT_AVAILABLE] Configuration spark.mlflow.modelRegistryUri is not available. SQLSTATE: 42K0I\", grpc_status:13}\"\n>", "stacktrace": ["Traceback (most recent call last):", "  File \"/Users/saroj.venkatesh/ai-dev-kit/.venv/lib/python3.11/site-packages/pyspark/sql/connect/client/core.py\", line 2042, in config", "    resp = self._stub.Config(req, metadata=self.metadata())", "           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^", "  File \"/Users/saroj.venkatesh/ai-dev-kit/.venv/lib/python3.11/site-packages/grpc

🏃 View run xgboost at: https://fevm-sv-ai-builder-workspace.cloud.databricks.com/ml/experiments/125118335144552/runs/03089afc5f1b467393705ae703377aa3
🧪 View experiment at: https://fevm-sv-ai-builder-workspace.cloud.databricks.com/ml/experiments/125118335144552

Best model: random_forest  (ROC-AUC: 0.7622)

Classification report (random_forest @ 0.5 threshold):
              precision    recall  f1-score   support

    Retained       0.71      0.68      0.69       757
     Churned       0.66      0.70      0.68       677

    accuracy                           0.69      1434
   macro avg       0.69      0.69      0.69      1434
weighted avg       0.69      0.69      0.69      1434



## 5. Feature Importance (SHAP)

SHAP values explain *why* the model makes each prediction -- critical for making churn scores actionable.
The summary plot shows which features drive churn predictions most strongly across the test set.

In [11]:
import shap
import matplotlib
matplotlib.use('Agg')   # non-interactive backend for Cursor
import matplotlib.pyplot as plt

# Use TreeExplainer (fast for tree-based models)
explainer   = shap.TreeExplainer(best_model)
shap_sample = X_test.sample(min(500, len(X_test)), random_state=42)
shap_values = explainer.shap_values(shap_sample)

# Normalise shap_values across SHAP versions and model types:
#   - list of 2 arrays  → LightGBM/RF binary old API → take index 1 (churn class)
#   - ndarray 3-D       → (n_samples, n_features, n_classes) → slice [:,:,1]
#   - ndarray 2-D       → already (n_samples, n_features)    → use as-is
#   - shap.Explanation  → extract .values
if isinstance(shap_values, list):
    sv = np.array(shap_values[1])
elif hasattr(shap_values, 'values'):          # shap.Explanation object
    sv = shap_values.values
    if sv.ndim == 3:
        sv = sv[:, :, 1]
elif isinstance(shap_values, np.ndarray):
    if shap_values.ndim == 3:
        sv = shap_values[:, :, 1]
    else:
        sv = shap_values
else:
    sv = np.array(shap_values)

# Bar chart: mean absolute SHAP per feature
mean_shap = pd.Series(np.abs(sv).mean(axis=0), index=FEATURE_COLS).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(8, 7))
mean_shap.tail(15).plot(kind='barh', ax=ax, color='#1b6ca8')
ax.set_title(f'Top 15 Churn Drivers (SHAP) -- {best_name}', fontsize=13)
ax.set_xlabel('Mean |SHAP value|')
plt.tight_layout()
plt.savefig('shap_importance.png', dpi=120)
plt.show()
print("Saved: shap_importance.png")

# Print top 10 ranked drivers
print("\nTop 10 churn drivers (ranked by impact):")
for i, (feat, val) in enumerate(mean_shap.tail(10).iloc[::-1].items(), 1):
    print(f"  {i:>2}. {feat:<30} {val:.4f}")

Saved: shap_importance.png

Top 10 churn drivers (ranked by impact):
   1. is_enterprise                  0.0514
   2. subscription_tier              0.0484
   3. total_ltv                      0.0470
   4. avg_order_value                0.0425
   5. lifetime_orders                0.0146
   6. days_since_last_order          0.0101
   7. orders_prev_30d                0.0069
   8. account_age_days               0.0042
   9. product_type                   0.0040
  10. country                        0.0039


## 6. Register Best Model + Score All Users

Register the winning model in Unity Catalog and write a churn score for every active user
back to `saas_analysis_sandbox.churn_scores` -- ready for CS team consumption.

In [12]:
import mlflow.sklearn
from mlflow.models import infer_signature

MODEL_UC_PATH = f"{CATALOG}.{SANDBOX}.churn_model"

# Compute signature from held-out test set
_y_prob_best = best_model.predict_proba(X_test)[:, 1]
_signature   = infer_signature(X_test, _y_prob_best)

with mlflow.start_run(run_name=f"{best_name}_registered"):
    mlflow.set_tag("champion", "true")
    mlflow.log_metrics({
        "roc_auc": round(results[best_name]['roc_auc'], 4),
        "pr_auc":  round(results[best_name]['pr_auc'],  4),
    })
    mlflow.sklearn.log_model(
        best_model,
        name="churn_model",
        signature=_signature,
        registered_model_name=MODEL_UC_PATH,
    )
    print(f"Model registered: {MODEL_UC_PATH}")

# Score all users and write to sandbox
all_features = df[FEATURE_COLS].copy()
churn_prob   = best_model.predict_proba(all_features)[:, 1]

# include_lowest=True makes the leftmost bin [0, 0.3] (closed on left) so
# users with probability == 0.0 get 'Low' instead of NaN.
risk_tier = pd.cut(
    churn_prob,
    bins=[0, 0.3, 0.6, 1.0],
    labels=['Low', 'Medium', 'High'],
    include_lowest=True,
).astype(str)

scores_df = pd.DataFrame({
    'user_id':           df['user_id'].astype(str),
    'churn_probability': churn_prob.round(4).astype(float),
    'churn_risk_tier':   risk_tier,
    'actual_churned':    df['churned'].astype(int),
})

scores_spark = spark.createDataFrame(scores_df)
scores_spark.write.mode('overwrite').saveAsTable(f"{CATALOG}.{SANDBOX}.churn_scores")

print(f"\nChurn scores written to: {CATALOG}.{SANDBOX}.churn_scores")
spark.sql(f"""
    SELECT churn_risk_tier, COUNT(*) AS users,
           ROUND(AVG(CAST(churn_probability AS DOUBLE)), 3) AS avg_prob
    FROM {CATALOG}.{SANDBOX}.churn_scores
    GROUP BY churn_risk_tier ORDER BY avg_prob DESC
""").show()

{"ts": "2026-02-26 14:56:12,416", "level": "ERROR", "logger": "pyspark.sql.connect.client.logging", "msg": "GRPC Error received", "context": {}, "exception": {"class": "_InactiveRpcError", "msg": "<_InactiveRpcError of RPC that terminated with:\n\tstatus = StatusCode.INTERNAL\n\tdetails = \"[CONFIG_NOT_AVAILABLE] Configuration spark.mlflow.modelRegistryUri is not available. SQLSTATE: 42K0I\"\n\tdebug_error_string = \"UNKNOWN:Error received from peer  {grpc_status:13, grpc_message:\"[CONFIG_NOT_AVAILABLE] Configuration spark.mlflow.modelRegistryUri is not available. SQLSTATE: 42K0I\"}\"\n>", "stacktrace": ["Traceback (most recent call last):", "  File \"/Users/saroj.venkatesh/ai-dev-kit/.venv/lib/python3.11/site-packages/pyspark/sql/connect/client/core.py\", line 2042, in config", "    resp = self._stub.Config(req, metadata=self.metadata())", "           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^", "  File \"/Users/saroj.venkatesh/ai-dev-kit/.venv/lib/python3.11/site-packages/grpc

Uploading artifacts:   0%|          | 0/5 [00:00<?, ?it/s]

{"ts": "2026-02-26 14:56:59,832", "level": "ERROR", "logger": "pyspark.sql.connect.client.logging", "msg": "GRPC Error received", "context": {}, "exception": {"class": "_InactiveRpcError", "msg": "<_InactiveRpcError of RPC that terminated with:\n\tstatus = StatusCode.INTERNAL\n\tdetails = \"[CONFIG_NOT_AVAILABLE] Configuration spark.mlflow.modelRegistryUri is not available. SQLSTATE: 42K0I\"\n\tdebug_error_string = \"UNKNOWN:Error received from peer  {grpc_status:13, grpc_message:\"[CONFIG_NOT_AVAILABLE] Configuration spark.mlflow.modelRegistryUri is not available. SQLSTATE: 42K0I\"}\"\n>", "stacktrace": ["Traceback (most recent call last):", "  File \"/Users/saroj.venkatesh/ai-dev-kit/.venv/lib/python3.11/site-packages/pyspark/sql/connect/client/core.py\", line 2042, in config", "    resp = self._stub.Config(req, metadata=self.metadata())", "           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^", "  File \"/Users/saroj.venkatesh/ai-dev-kit/.venv/lib/python3.11/site-packages/grpc

Model registered: sv_ai_builder_workspace_catalog.saas_analysis_sandbox.churn_model


{"ts": "2026-02-26 14:57:04,451", "level": "ERROR", "logger": "pyspark.sql.connect.client.logging", "msg": "GRPC Error received", "context": {}, "exception": {"class": "_InactiveRpcError", "msg": "<_InactiveRpcError of RPC that terminated with:\n\tstatus = StatusCode.INTERNAL\n\tdetails = \"[CONFIG_NOT_AVAILABLE] Configuration spark.mlflow.modelRegistryUri is not available. SQLSTATE: 42K0I\"\n\tdebug_error_string = \"UNKNOWN:Error received from peer  {grpc_message:\"[CONFIG_NOT_AVAILABLE] Configuration spark.mlflow.modelRegistryUri is not available. SQLSTATE: 42K0I\", grpc_status:13}\"\n>", "stacktrace": ["Traceback (most recent call last):", "  File \"/Users/saroj.venkatesh/ai-dev-kit/.venv/lib/python3.11/site-packages/pyspark/sql/connect/client/core.py\", line 2042, in config", "    resp = self._stub.Config(req, metadata=self.metadata())", "           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^", "  File \"/Users/saroj.venkatesh/ai-dev-kit/.venv/lib/python3.11/site-packages/grpc

🏃 View run random_forest_registered at: https://fevm-sv-ai-builder-workspace.cloud.databricks.com/ml/experiments/125118335144552/runs/7004947d8b604db88f2707f34c9d88ce
🧪 View experiment at: https://fevm-sv-ai-builder-workspace.cloud.databricks.com/ml/experiments/125118335144552

Churn scores written to: sv_ai_builder_workspace_catalog.saas_analysis_sandbox.churn_scores
+---------------+-----+--------+
|churn_risk_tier|users|avg_prob|
+---------------+-----+--------+
|           High| 2540|   0.706|
|         Medium| 2563|   0.479|
|            Low| 2066|   0.173|
+---------------+-----+--------+

